<a href="https://colab.research.google.com/github/greek-nlp/benchmark/blob/main/suite_benchmark_colab_monte_carlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Greek NLP Benchmark Suite in Colab

This notebook is the Colab entrypoint for the benchmark tasks in this repository.

Current state:
- The notebook sets up Ollama inside Colab.
- It organizes the full task suite in one place.
- The repository now includes a shared runner in `suite_benchmark.py`.
- Direct shared-runner tasks currently include GEC, toxicity, machine translation, intent classification, and summarization.
- The remaining tasks still point to the task-specific notebooks already present in the repo.


## Quick Start

Run the notebook top to bottom:
1. Setup cells
2. Suite configuration
3. Model pull cell
4. Task overview
5. GEC benchmark cells

Important: in Ollama, use `llama3.1:8b`, not `llama3.1:8b-instruct`.


## Repo Setup

Open this notebook from the repository in Colab, or clone/upload the repository files into the runtime before continuing.


## 1. Environment Setup

In [1]:
!apt-get -qq update
!apt-get -qq install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!pip -q install pandas pywer openpyxl datasets scikit-learn

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
print("Started Ollama server with PID", ollama_process.pid)

Started Ollama server with PID 3168


## 2. Suite Configuration

In [5]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
print("Working directory:", project_root)
print("Python executable:", sys.executable)

MODEL_PRESETS = {
    "t4": [
        "qwen2.5:3b",
        "gemma2:2b",
        "mistral:7b",
        "qwen2.5:7b-instruct",
    ],
    "high_memory": [
        "llama3.1:8b",
        "aya-expanse:8b",
        "gemma2:9b",
        "qwen2.5:14b",
    ],
}

TASKS = {
    "toxicity": {
        "datasets": ["OGTD / OffenseEval Greek"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "gec": {
        "datasets": ["GNC / Korre"],
        "entrypoints": ["suite_benchmark_colab_monte_carlo.ipynb", "gec_benchmark_notebook.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "machine_translation": {
        "datasets": ["PGV"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "summarization": {
        "datasets": ["GreekLegalSum"],
        "entrypoints": ["nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "intent_classification": {
        "datasets": ["UniWay GR"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "ner": {
        "datasets": ["elNER", "ner_nel_greek_dataset", "UniWay GR"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_access_data.ipynb"],
        "runner": "notebook",
    },
    "pos_tagging": {
        "datasets": ["UD Greek GDT"],
        "entrypoints": ["nlp_gr_access_data.ipynb"],
        "runner": "notebook",
    },
    "clustering": {
        "datasets": ["Greek Legal Code"],
        "entrypoints": ["nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
    "authorship_attribution": {
        "datasets": ["author datasets in repo notebooks"],
        "entrypoints": ["gr_attribution_zeroshot.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
}

MODEL_PRESET = "t4"
MODELS = MODEL_PRESETS[MODEL_PRESET]
ACTIVE_TASKS = list(TASKS)

SAMPLE_SIZE = 100
NUM_REPEATS = 5
RANDOM_STATE = 42
RESUME_REPEATS = True
OUTPUT_ROOT = Path("results/colab_suite")

print("Model preset:", MODEL_PRESET)
print("Models:", MODELS)
print("Tasks:", ACTIVE_TASKS)
print("Sample size:", SAMPLE_SIZE)
print("Repeats per task:", NUM_REPEATS)
print("Resume existing repeats:", RESUME_REPEATS)


Working directory: /content
Python executable: /usr/bin/python3
Model preset: t4
Models: ['qwen2.5:3b', 'gemma2:2b', 'mistral:7b', 'qwen2.5:7b-instruct']
Tasks: ['toxicity', 'gec', 'machine_translation', 'summarization', 'intent_classification', 'ner', 'pos_tagging', 'clustering', 'authorship_attribution']
Sample size: 100
Repeats per task: 5
Resume existing repeats: True


In [6]:
for model in MODELS:
    print(f"Pulling {model}...")
    subprocess.run(["ollama", "pull", model], check=True)

Pulling qwen2.5:3b...
Pulling gemma2:2b...
Pulling mistral:7b...
Pulling qwen2.5:7b-instruct...


In [7]:
!ollama list

NAME                   ID              SIZE      MODIFIED               
qwen2.5:7b-instruct    845dbda0ea48    4.7 GB    Less than a second ago    
mistral:7b             6577803aa9a0    4.4 GB    28 seconds ago            
gemma2:2b              8ccf136fdd52    1.6 GB    57 seconds ago            
qwen2.5:3b             357c53fb659c    1.9 GB    About a minute ago        


## 3. Task Overview

In [8]:
task_rows = []
for task_name, info in TASKS.items():
    task_rows.append(
        {
            "task": task_name,
            "runner": info["runner"],
            "datasets": ", ".join(info["datasets"]),
            "entrypoints": ", ".join(info["entrypoints"]),
        }
    )

pd.DataFrame(task_rows)

,task,runner,datasets,entrypoints
0,toxicity,shared_runner,OGTD / OffenseEval Greek,"air_gr_nlp_benchmark.ipynb, nlp_gr_experiments..."
1,gec,shared_runner,GNC / Korre,"suite_benchmark_colab_monte_carlo.ipynb, gec_b..."
2,machine_translation,shared_runner,PGV,"air_gr_nlp_benchmark.ipynb, nlp_gr_experiments..."
3,summarization,shared_runner,GreekLegalSum,nlp_gr_experiments.ipynb
4,intent_classification,shared_runner,UniWay GR,"air_gr_nlp_benchmark.ipynb, nlp_gr_experiments..."
5,ner,notebook,"elNER, ner_nel_greek_dataset, UniWay GR","air_gr_nlp_benchmark.ipynb, nlp_gr_access_data..."
6,pos_tagging,notebook,UD Greek GDT,nlp_gr_access_data.ipynb
7,clustering,notebook,Greek Legal Code,nlp_gr_experiments.ipynb
8,authorship_attribution,notebook,author datasets in repo notebooks,"gr_attribution_zeroshot.ipynb, nlp_gr_experime..."


## 4. Notebook Routing for Non-Shared Tasks

The shared runner currently covers GEC, toxicity, machine translation, intent classification, and summarization.

For the remaining tasks, start from these notebooks:
- NER: `air_gr_nlp_benchmark.ipynb`, `nlp_gr_access_data.ipynb`
- POS tagging: `nlp_gr_access_data.ipynb`
- Clustering: `nlp_gr_experiments.ipynb`
- Authorship attribution: `gr_attribution_zeroshot.ipynb`, `nlp_gr_experiments.ipynb`


## 5. Run the Shared Benchmark Suite and Inspect Results Per Task

In [9]:
import os

repo_url = "https://github.com/greek-nlp/benchmark.git"
repo_dir = "benchmark"

# Clone the repository if it doesn't already exist
if not os.path.exists(repo_dir):
    print(f"Cloning {repo_url} into {repo_dir}...")
    !git clone {repo_url}
    # Change current working directory to the cloned repository
    %cd {repo_dir}
    print(f"Changed current working directory to {os.getcwd()}")
else:
    print(f"Repository {repo_dir} already exists. Pulling latest changes...")
    # Change into the directory and pull latest changes
    %cd {repo_dir}
    !git pull
    print(f"Current working directory is {os.getcwd()}")

Cloning https://github.com/greek-nlp/benchmark.git into benchmark...
Cloning into 'benchmark'...
remote: Enumerating objects: 345, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 345 (delta 6), reused 2 (delta 2), pack-reused 336 (from 2)
Receiving objects: 100% (345/345), 20.92 MiB | 60.34 MiB/s, done.
Resolving deltas: 100% (197/197), done.
/content/benchmark
Changed current working directory to /content/benchmark


In [10]:
import sys

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from benchmark_suite import GenerationConfig, list_tasks, run_task, save_run_outputs

def aggregate_repeated_summaries(summary_by_repeat):
    repeated_summary = pd.concat(summary_by_repeat, ignore_index=True)
    key_columns = [column for column in ["task", "model"] if column in repeated_summary.columns]
    numeric_columns = [
        column
        for column in repeated_summary.columns
        if column not in key_columns + ["repeat"] and pd.api.types.is_numeric_dtype(repeated_summary[column])
    ]

    aggregated = repeated_summary[key_columns].drop_duplicates().sort_values(key_columns).reset_index(drop=True)
    grouped = repeated_summary.groupby(key_columns, sort=False)

    for column in numeric_columns:
        aggregated[f"{column}_mean"] = grouped[column].mean().to_numpy()
        aggregated[f"{column}_sem"] = grouped[column].sem(ddof=1).fillna(0.0).to_numpy()

    return aggregated, repeated_summary

def load_repeat_outputs(repeat_output_dir, task_name):
    summary = pd.read_csv(repeat_output_dir / f"{task_name}_summary.csv")
    raw = pd.read_csv(repeat_output_dir / f"{task_name}_predictions.csv")
    return summary, raw

SHARED_TASKS = list_tasks()
CONFIG = GenerationConfig(
    temperature=0.0,
    num_predict=256,
    timeout_seconds=300,
)

print("Running shared-runner tasks:", SHARED_TASKS)
print("Models:", MODELS)
print("Output root:", OUTPUT_ROOT)
print("Sample size:", SAMPLE_SIZE)
print("Repeats per task:", NUM_REPEATS)
print("Resume existing repeats:", RESUME_REPEATS)


Running shared-runner tasks: ['gec', 'intent', 'mt_eng', 'mt_fas', 'mt_jpn', 'summarization', 'toxicity']
Models: ['qwen2.5:3b', 'gemma2:2b', 'mistral:7b', 'qwen2.5:7b-instruct']
Output root: results/colab_suite
Sample size: 100
Repeats per task: 5
Resume existing repeats: True


In [11]:
task_summaries = {}
task_predictions = {}
task_repeat_summaries = {}
combined_summaries = []

for task_name in SHARED_TASKS:
    print(f"Running task: {task_name}")
    repeat_summaries = []
    repeat_predictions = []

    for repeat_index in range(NUM_REPEATS):
        repeat_number = repeat_index + 1
        repeat_seed = RANDOM_STATE + repeat_index
        repeat_output_dir = OUTPUT_ROOT / task_name / f"repeat_{repeat_number:02d}"

        if RESUME_REPEATS and (repeat_output_dir / f"{task_name}_summary.csv").exists():
            print(f"  Reusing repeat {repeat_number}/{NUM_REPEATS} from {repeat_output_dir}")
            summary, raw = load_repeat_outputs(repeat_output_dir, task_name)
        else:
            print(f"  Repeat {repeat_number}/{NUM_REPEATS} with seed {repeat_seed}")
            summary, raw = run_task(
                task_name=task_name,
                models=MODELS,
                sample_size=SAMPLE_SIZE,
                random_state=repeat_seed,
                config=CONFIG,
            )
            save_run_outputs(summary, raw, repeat_output_dir, task_name)
            print(f"  Saved repeat outputs to {repeat_output_dir}")

        summary = summary.copy()
        raw = raw.copy()
        summary["repeat"] = repeat_number
        raw["repeat"] = repeat_number
        repeat_summaries.append(summary)
        repeat_predictions.append(raw)

    aggregated_summary, repeated_summary = aggregate_repeated_summaries(repeat_summaries)
    repeated_predictions = pd.concat(repeat_predictions, ignore_index=True)
    task_output_dir = OUTPUT_ROOT / task_name
    task_output_dir.mkdir(parents=True, exist_ok=True)
    aggregated_summary.to_csv(task_output_dir / f"{task_name}_summary_with_sem.csv", index=False)
    repeated_summary.to_csv(task_output_dir / f"{task_name}_repeat_summaries.csv", index=False)
    repeated_predictions.to_csv(task_output_dir / f"{task_name}_repeat_predictions.csv", index=False)

    task_summaries[task_name] = aggregated_summary
    task_predictions[task_name] = repeated_predictions
    task_repeat_summaries[task_name] = repeated_summary
    combined_summaries.append(aggregated_summary)
    print(f"Saved aggregated outputs to {task_output_dir}")

combined_summary = pd.concat(combined_summaries, ignore_index=True)
combined_summary_path = OUTPUT_ROOT / "all_tasks_summary_with_sem.csv"
combined_summary.to_csv(combined_summary_path, index=False)
print(f"Saved combined summary to {combined_summary_path}")

combined_summary


Running task: gec
  Repeat 1/5 with seed 42
  Saved repeat outputs to results/colab_suite/gec/repeat_01
  Repeat 2/5 with seed 43
  Saved repeat outputs to results/colab_suite/gec/repeat_02
  Repeat 3/5 with seed 44
  Saved repeat outputs to results/colab_suite/gec/repeat_03
  Repeat 4/5 with seed 45
  Saved repeat outputs to results/colab_suite/gec/repeat_04
  Repeat 5/5 with seed 46
  Saved repeat outputs to results/colab_suite/gec/repeat_05
Saved aggregated outputs to results/colab_suite/gec
Running task: intent
  Repeat 1/5 with seed 42
  Saved repeat outputs to results/colab_suite/intent/repeat_01
  Repeat 2/5 with seed 43
  Saved repeat outputs to results/colab_suite/intent/repeat_02
  Repeat 3/5 with seed 44
  Saved repeat outputs to results/colab_suite/intent/repeat_03
  Repeat 4/5 with seed 45
  Saved repeat outputs to results/colab_suite/intent/repeat_04
  Repeat 5/5 with seed 46
  Saved repeat outputs to results/colab_suite/intent/repeat_05
Saved aggregated outputs to result

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/153 [00:00<?, ?B/s]

hugginface_dataset.csv:   0%|          | 0.00/289M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8395 [00:00<?, ? examples/s]

...

offenseval-gr-test-v1.tsv:   0%|          | 0.00/301k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

  Saved repeat outputs to results/colab_suite/toxicity/repeat_01
  Repeat 2/5 with seed 43
  Saved repeat outputs to results/colab_suite/toxicity/repeat_02
  Repeat 3/5 with seed 44
  Saved repeat outputs to results/colab_suite/toxicity/repeat_03
  Repeat 4/5 with seed 45
  Saved repeat outputs to results/colab_suite/toxicity/repeat_04
  Repeat 5/5 with seed 46
  Saved repeat outputs to results/colab_suite/toxicity/repeat_05
Saved aggregated outputs to results/colab_suite/toxicity
Saved combined summary to results/colab_suite/all_tasks_summary_with_sem.csv


,task,model,samples_mean,samples_sem,exact_match_mean,exact_match_sem,wer_vs_reference_mean,wer_vs_reference_sem,cer_vs_reference_mean,cer_vs_reference_sem,...,cer_vs_original_mean,cer_vs_original_sem,changed_input_rate_mean,changed_input_rate_sem,avg_latency_seconds_mean,avg_latency_seconds_sem,accuracy_mean,accuracy_sem,macro_f1_mean,macro_f1_sem
0,gec,gemma2:2b,100.0,0.0,0.036,0.005099,12.178039,0.419360,5.962845,0.226838,...,5.394651,0.214134,0.820,0.010488,2.671117,0.125073,NaN,NaN,NaN,NaN
1,gec,mistral:7b,100.0,0.0,0.068,0.003742,13.393058,0.416139,6.573561,0.216682,...,6.264826,0.207213,0.948,0.004899,6.023327,0.075473,NaN,NaN,NaN,NaN
2,gec,qwen2.5:3b,100.0,0.0,0.134,0.006782,13.500445,0.677311,8.867185,0.833394,...,8.751081,0.812815,0.866,0.009274,1.310764,0.108196,NaN,NaN,NaN,NaN
3,gec,qwen2.5:7b-instruct,100.0,0.0,0.000,0.000000,66.226902,1.563838,57.130962,1.699570,...,56.702982,1.720469,1.000,0.000000,7.814076,0.079279,NaN,NaN,NaN,NaN
4,intent,gemma2:2b,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.536295,0.021755,0.756,0.021354,0.586948,0.014998
5,intent,mistral:7b,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.531371,0.016062,0.694,0.014353,0.276946,0.033627
6,intent,qwen2.5:3b,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.371796,0.005827,0.618,0.015620,0.619503,0.014240
7,intent,qwen2.5:7b-instruct,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.346548,0.001179,0.476,0.002449,0.288883,0.003721
8,mt_eng,gemma2:2b,100.0,0.0,NaN,NaN,68.400119,4.248258,48.942422,2.399803,...,NaN,NaN,NaN,NaN,1.420048,0.015895,NaN,NaN,NaN,NaN
9,mt_eng,mistral:7b,100.0,0.0,NaN,NaN,66.735634,3.301298,47.357103,2.040556,...,NaN,NaN,NaN,NaN,0.677774,0.003792,NaN,NaN,NaN,NaN


In [12]:
METRIC_PREFERENCES = [
    "exact_match_mean",
    "accuracy_mean",
    "macro_f1_mean",
    "wer_vs_reference_mean",
    "cer_vs_reference_mean",
    "avg_latency_seconds_mean",
]

LOWER_IS_BETTER = {"wer_vs_reference_mean", "cer_vs_reference_mean", "avg_latency_seconds_mean"}

performance_rows = []

for task_name in SHARED_TASKS:
    summary = task_summaries[task_name].copy()
    print(f"\n=== {task_name} ===")
    display(summary)

    metric = next((column for column in METRIC_PREFERENCES if column in summary.columns), None)
    if metric is None:
        continue

    sem_column = metric.replace("_mean", "_sem")
    ascending = metric in LOWER_IS_BETTER
    ranked = summary.sort_values(metric, ascending=ascending).reset_index(drop=True)
    best_row = ranked.iloc[0]
    performance_rows.append(
        {
            "task": task_name,
            "primary_metric": metric,
            "best_model": best_row["model"],
            "best_score_mean": best_row[metric],
            "best_score_sem": best_row[sem_column] if sem_column in best_row.index else 0.0,
        }
    )

performance_by_task = pd.DataFrame(performance_rows)
print("\n=== Best Performance Per Task ===")
display(performance_by_task)

combined_summary



=== gec ===


,task,model,samples_mean,samples_sem,exact_match_mean,exact_match_sem,wer_vs_reference_mean,wer_vs_reference_sem,cer_vs_reference_mean,cer_vs_reference_sem,wer_vs_original_mean,wer_vs_original_sem,cer_vs_original_mean,cer_vs_original_sem,changed_input_rate_mean,changed_input_rate_sem,avg_latency_seconds_mean,avg_latency_seconds_sem
0,gec,gemma2:2b,100.0,0.0,0.036,0.005099,12.178039,0.419360,5.962845,0.226838,10.004364,0.385223,5.394651,0.214134,0.820,0.010488,2.671117,0.125073
1,gec,mistral:7b,100.0,0.0,0.068,0.003742,13.393058,0.416139,6.573561,0.216682,12.717744,0.386609,6.264826,0.207213,0.948,0.004899,6.023327,0.075473
2,gec,qwen2.5:3b,100.0,0.0,0.134,0.006782,13.500445,0.677311,8.867185,0.833394,13.006205,0.650211,8.751081,0.812815,0.866,0.009274,1.310764,0.108196
3,gec,qwen2.5:7b-instruct,100.0,0.0,0.000,0.000000,66.226902,1.563838,57.130962,1.699570,64.842266,1.648256,56.702982,1.720469,1.000,0.000000,7.814076,0.079279



=== intent ===


,task,model,samples_mean,samples_sem,accuracy_mean,accuracy_sem,macro_f1_mean,macro_f1_sem,avg_latency_seconds_mean,avg_latency_seconds_sem
0,intent,gemma2:2b,100.0,0.0,0.756,0.021354,0.586948,0.014998,0.536295,0.021755
1,intent,mistral:7b,100.0,0.0,0.694,0.014353,0.276946,0.033627,0.531371,0.016062
2,intent,qwen2.5:3b,100.0,0.0,0.618,0.015620,0.619503,0.014240,0.371796,0.005827
3,intent,qwen2.5:7b-instruct,100.0,0.0,0.476,0.002449,0.288883,0.003721,0.346548,0.001179



=== mt_eng ===


...


=== toxicity ===


,task,model,samples_mean,samples_sem,accuracy_mean,accuracy_sem,macro_f1_mean,macro_f1_sem,avg_latency_seconds_mean,avg_latency_seconds_sem
0,toxicity,gemma2:2b,100.0,0.0,0.0,0.0,0.0,0.0,0.273580,0.001016
1,toxicity,mistral:7b,100.0,0.0,0.0,0.0,0.0,0.0,0.309319,0.004727
2,toxicity,qwen2.5:3b,100.0,0.0,0.0,0.0,0.0,0.0,0.236190,0.006385
3,toxicity,qwen2.5:7b-instruct,100.0,0.0,0.0,0.0,0.0,0.0,0.347027,0.002354



=== Best Performance Per Task ===


,task,primary_metric,best_model,best_score_mean,best_score_sem
0,gec,exact_match_mean,qwen2.5:3b,0.134000,0.006782
1,intent,accuracy_mean,gemma2:2b,0.756000,0.021354
2,mt_eng,wer_vs_reference_mean,mistral:7b,66.735634,3.301298
3,mt_fas,wer_vs_reference_mean,mistral:7b,110.541547,0.871143
4,mt_jpn,wer_vs_reference_mean,gemma2:2b,153.683810,7.606168
5,summarization,wer_vs_reference_mean,gemma2:2b,367.845985,26.608786
6,toxicity,accuracy_mean,gemma2:2b,0.000000,0.000000


,task,model,samples_mean,samples_sem,exact_match_mean,exact_match_sem,wer_vs_reference_mean,wer_vs_reference_sem,cer_vs_reference_mean,cer_vs_reference_sem,...,cer_vs_original_mean,cer_vs_original_sem,changed_input_rate_mean,changed_input_rate_sem,avg_latency_seconds_mean,avg_latency_seconds_sem,accuracy_mean,accuracy_sem,macro_f1_mean,macro_f1_sem
0,gec,gemma2:2b,100.0,0.0,0.036,0.005099,12.178039,0.419360,5.962845,0.226838,...,5.394651,0.214134,0.820,0.010488,2.671117,0.125073,NaN,NaN,NaN,NaN
1,gec,mistral:7b,100.0,0.0,0.068,0.003742,13.393058,0.416139,6.573561,0.216682,...,6.264826,0.207213,0.948,0.004899,6.023327,0.075473,NaN,NaN,NaN,NaN
2,gec,qwen2.5:3b,100.0,0.0,0.134,0.006782,13.500445,0.677311,8.867185,0.833394,...,8.751081,0.812815,0.866,0.009274,1.310764,0.108196,NaN,NaN,NaN,NaN
3,gec,qwen2.5:7b-instruct,100.0,0.0,0.000,0.000000,66.226902,1.563838,57.130962,1.699570,...,56.702982,1.720469,1.000,0.000000,7.814076,0.079279,NaN,NaN,NaN,NaN
4,intent,gemma2:2b,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.536295,0.021755,0.756,0.021354,0.586948,0.014998
5,intent,mistral:7b,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.531371,0.016062,0.694,0.014353,0.276946,0.033627
6,intent,qwen2.5:3b,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.371796,0.005827,0.618,0.015620,0.619503,0.014240
7,intent,qwen2.5:7b-instruct,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.346548,0.001179,0.476,0.002449,0.288883,0.003721
8,mt_eng,gemma2:2b,100.0,0.0,NaN,NaN,68.400119,4.248258,48.942422,2.399803,...,NaN,NaN,NaN,NaN,1.420048,0.015895,NaN,NaN,NaN,NaN
9,mt_eng,mistral:7b,100.0,0.0,NaN,NaN,66.735634,3.301298,47.357103,2.040556,...,NaN,NaN,NaN,NaN,0.677774,0.003792,NaN,NaN,NaN,NaN


In [13]:
METRIC_PREFERENCES = [
    "exact_match",
    "accuracy",
    "macro_f1",
    "wer_vs_reference",
    "cer_vs_reference",
    "avg_latency_seconds",
]

LOWER_IS_BETTER = {"wer_vs_reference", "cer_vs_reference", "avg_latency_seconds"}

performance_rows = []

for task_name in SHARED_TASKS:
    summary = task_summaries[task_name].copy()
    print(f"\n=== {task_name} ===")
    display(summary)

    metric = next((column for column in METRIC_PREFERENCES if column in summary.columns), None)
    if metric is None:
        continue

    ascending = metric in LOWER_IS_BETTER
    ranked = summary.sort_values(metric, ascending=ascending).reset_index(drop=True)
    best_row = ranked.iloc[0]
    performance_rows.append(
        {
            "task": task_name,
            "primary_metric": metric,
            "best_model": best_row["model"],
            "best_score": best_row[metric],
            "samples": best_row.get("samples"),
        }
    )

performance_by_task = pd.DataFrame(performance_rows)
print("\n=== Best Performance Per Task ===")
display(performance_by_task)

combined_summary



=== gec ===


,task,model,samples_mean,samples_sem,exact_match_mean,exact_match_sem,wer_vs_reference_mean,wer_vs_reference_sem,cer_vs_reference_mean,cer_vs_reference_sem,wer_vs_original_mean,wer_vs_original_sem,cer_vs_original_mean,cer_vs_original_sem,changed_input_rate_mean,changed_input_rate_sem,avg_latency_seconds_mean,avg_latency_seconds_sem
0,gec,gemma2:2b,100.0,0.0,0.036,0.005099,12.178039,0.419360,5.962845,0.226838,10.004364,0.385223,5.394651,0.214134,0.820,0.010488,2.671117,0.125073
1,gec,mistral:7b,100.0,0.0,0.068,0.003742,13.393058,0.416139,6.573561,0.216682,12.717744,0.386609,6.264826,0.207213,0.948,0.004899,6.023327,0.075473
2,gec,qwen2.5:3b,100.0,0.0,0.134,0.006782,13.500445,0.677311,8.867185,0.833394,13.006205,0.650211,8.751081,0.812815,0.866,0.009274,1.310764,0.108196
3,gec,qwen2.5:7b-instruct,100.0,0.0,0.000,0.000000,66.226902,1.563838,57.130962,1.699570,64.842266,1.648256,56.702982,1.720469,1.000,0.000000,7.814076,0.079279



=== intent ===


,task,model,samples_mean,samples_sem,accuracy_mean,accuracy_sem,macro_f1_mean,macro_f1_sem,avg_latency_seconds_mean,avg_latency_seconds_sem
0,intent,gemma2:2b,100.0,0.0,0.756,0.021354,0.586948,0.014998,0.536295,0.021755
1,intent,mistral:7b,100.0,0.0,0.694,0.014353,0.276946,0.033627,0.531371,0.016062
2,intent,qwen2.5:3b,100.0,0.0,0.618,0.015620,0.619503,0.014240,0.371796,0.005827
3,intent,qwen2.5:7b-instruct,100.0,0.0,0.476,0.002449,0.288883,0.003721,0.346548,0.001179



=== mt_eng ===


...


=== toxicity ===


,task,model,samples_mean,samples_sem,accuracy_mean,accuracy_sem,macro_f1_mean,macro_f1_sem,avg_latency_seconds_mean,avg_latency_seconds_sem
0,toxicity,gemma2:2b,100.0,0.0,0.0,0.0,0.0,0.0,0.273580,0.001016
1,toxicity,mistral:7b,100.0,0.0,0.0,0.0,0.0,0.0,0.309319,0.004727
2,toxicity,qwen2.5:3b,100.0,0.0,0.0,0.0,0.0,0.0,0.236190,0.006385
3,toxicity,qwen2.5:7b-instruct,100.0,0.0,0.0,0.0,0.0,0.0,0.347027,0.002354



=== Best Performance Per Task ===


""


,task,model,samples_mean,samples_sem,exact_match_mean,exact_match_sem,wer_vs_reference_mean,wer_vs_reference_sem,cer_vs_reference_mean,cer_vs_reference_sem,...,cer_vs_original_mean,cer_vs_original_sem,changed_input_rate_mean,changed_input_rate_sem,avg_latency_seconds_mean,avg_latency_seconds_sem,accuracy_mean,accuracy_sem,macro_f1_mean,macro_f1_sem
0,gec,gemma2:2b,100.0,0.0,0.036,0.005099,12.178039,0.419360,5.962845,0.226838,...,5.394651,0.214134,0.820,0.010488,2.671117,0.125073,NaN,NaN,NaN,NaN
1,gec,mistral:7b,100.0,0.0,0.068,0.003742,13.393058,0.416139,6.573561,0.216682,...,6.264826,0.207213,0.948,0.004899,6.023327,0.075473,NaN,NaN,NaN,NaN
2,gec,qwen2.5:3b,100.0,0.0,0.134,0.006782,13.500445,0.677311,8.867185,0.833394,...,8.751081,0.812815,0.866,0.009274,1.310764,0.108196,NaN,NaN,NaN,NaN
3,gec,qwen2.5:7b-instruct,100.0,0.0,0.000,0.000000,66.226902,1.563838,57.130962,1.699570,...,56.702982,1.720469,1.000,0.000000,7.814076,0.079279,NaN,NaN,NaN,NaN
4,intent,gemma2:2b,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.536295,0.021755,0.756,0.021354,0.586948,0.014998
5,intent,mistral:7b,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.531371,0.016062,0.694,0.014353,0.276946,0.033627
6,intent,qwen2.5:3b,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.371796,0.005827,0.618,0.015620,0.619503,0.014240
7,intent,qwen2.5:7b-instruct,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.346548,0.001179,0.476,0.002449,0.288883,0.003721
8,mt_eng,gemma2:2b,100.0,0.0,NaN,NaN,68.400119,4.248258,48.942422,2.399803,...,NaN,NaN,NaN,NaN,1.420048,0.015895,NaN,NaN,NaN,NaN
9,mt_eng,mistral:7b,100.0,0.0,NaN,NaN,66.735634,3.301298,47.357103,2.040556,...,NaN,NaN,NaN,NaN,0.677774,0.003792,NaN,NaN,NaN,NaN
